# Lugares de Lisboa

## Html struct website
link: https://www.lisbongaycircuit.com/#


menu -> item -> sub-menu -> sub-menu item.

## Selenium e WebDriver Manager
I’ll use Python’s Selenium library instead, as some tests with Requests showed that Selenium performs better for scraping dynamic content.
 
In the next cell, I’ll import the librarys (which essentially means loading it into my code so I can use its tools) and set up a WebDriver instance for data collection.


In [2]:
#%config IPCompleter.greedy=True #autocomplete

In [3]:
import pandas as pd
from selenium import webdriver  # For automation, browser control, and pipeline tasks  
from webdriver_manager.chrome import ChromeDriverManager  # Automatically handles ChromeDriver setup (no manual downloads needed)  
from selenium.webdriver.chrome.service import Service  
from selenium.webdriver.common.by import By  
from selenium.webdriver.support.ui import WebDriverWait  
from selenium.webdriver.support import expected_conditions as EC  # Waits for elements before interacting (e.g., clicking)  
from selenium.webdriver.chrome.options import Options  # Handles browser settings 
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import TimeoutException 


## Criação de funções 

In [4]:
def create_browser():
    # configure adblocker (i'm not sure if i can do this, but this is an open-source adblocker i downloaded the source code from github)
    chrome_options = Options()
    #adblock_folder = r'D:\extensao_adblock\uBlock-master\platform\chromium'
    #chrome_options.add_argument(f'--load-extension={adblock_folder}')

    # creating browser
    service = Service(ChromeDriverManager().install())  # installs the driver for the current chrome version
    browser = webdriver.Chrome(service=service, options=chrome_options)
    return browser

## Scraping data from Lisbon Gay Circuit website

In [5]:
# collect texts from paragraphs (contains beaches data)
def collect_beaches_data(browser):
    beaches = []
    
    try:
        xpath = '//*[@id="single_cont"]/div/div'
        div = WebDriverWait(browser,10).until(EC.presence_of_element_located((By.XPATH, xpath)))
        paragraphs = div.find_elements(By.TAG_NAME, 'p')
        
        for p in paragraphs:
            print(p.text)
            if p.text:
                dictionary = {
                    'name': p.text.strip(),
                    'category': "BEACHES"
                }
                beaches.append(dictionary)
                print(dictionary)
    
    except TimeoutException:
        print("error: 'div' element not found on page")
   
    except Exception as e:
        print(e)
    
    return beaches

# collect categories and place names from site menu  
def collect_from_main_site(browser):
    data = []
    # skip these menu items (non-categories or beaches)
    skip_categories = ["BEACHES", "CONTACTS & AGENDA", "SHOPS & SERVICES", "TOURS & TRAVEL"]
   
    try:
        # site pattern: categories are 'li' with this specific class
        selector_category = 'li.menu-item-has-children'   
        categories = WebDriverWait(browser, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, selector_category)))
        
        # submenus require hover interaction
        actions = ActionChains(browser)
        
        for category in categories:
            try:
                # get category text
                category_name = category.find_element(By.XPATH, './a').text.strip()
                if not category_name or category_name in skip_categories:
                    continue
            
            except Exception as e:
                print(f"warning: couldn't read category. error: {e}") 
                continue
            
            try:
                # to show submenu
                actions.move_to_element(category).perform()
                selector_item = ".//ul[contains(@class, 'sub-menu')]//li"
                WebDriverWait(category, 2).until(EC.visibility_of_element_located((By.XPATH, selector_item)))
            
            except Exception as e:
                print(e)
                continue
            
            # submenu items follow this pattern
            selector_submenu_items = ".//ul[contains(@class, 'sub-menu')]//a"
            submenu_items = category.find_elements(By.XPATH, selector_submenu_items)
            
            for item in submenu_items:
                place_name = item.text.strip()
                print(place_name)
                if place_name:
                    dictionary = {
                        'name': place_name,
                        'category': category_name
                    }
                    data.append(dictionary)
                    print(dictionary)
    except Exception as e:
        print(f"error during collection: {e}")
    return data

# Main Function

In [6]:
# main function where we'll call other functions

if __name__ == "__main__":
    main_url = 'https://www.lisbongaycircuit.com/#'
    beach_url = 'https://www.lisbongaycircuit.com/praias-lgbt-friendly-em-portugal/'
    browser = None
    data = []
    
    try:
        # open browser on main url and call collection function
        browser = create_browser()
        browser.get(main_url)
        main_data = collect_from_main_site(browser)
        if main_data:
            data.extend(main_data)
        
        
        # go to beach url and call specific collection function
        browser.get(beach_url)
        beaches_data = collect_beaches_data(browser)
        if beaches_data:
            data.extend(beaches_data)
   
    except Exception as e:
        print(f"error during execution: {e}")
   
    finally:
        if browser:
            browser.quit()
    
    if data:
        # convert list to dataframe
        df = pd.DataFrame(data)
        print(df)

TRUMPS
{'name': 'TRUMPS', 'category': 'NIGHTCLUBS & PARTIES'}
FINALMENTE CLUB
{'name': 'FINALMENTE CLUB', 'category': 'NIGHTCLUBS & PARTIES'}
CONSTRUCTION LISBON CLUB
{'name': 'CONSTRUCTION LISBON CLUB', 'category': 'NIGHTCLUBS & PARTIES'}
PRESTIGE DANCECLUB – FARO
{'name': 'PRESTIGE DANCECLUB – FARO', 'category': 'NIGHTCLUBS & PARTIES'}
106
{'name': '106', 'category': 'BARS'}
BAR CRU
{'name': 'BAR CRU', 'category': 'BARS'}
CONSTRUCTION BAR – BAIRRO ALTO
{'name': 'CONSTRUCTION BAR – BAIRRO ALTO', 'category': 'BARS'}
ENTRETANTO ROOFTOP BAR
{'name': 'ENTRETANTO ROOFTOP BAR', 'category': 'BARS'}
TR3S
{'name': 'TR3S', 'category': 'BARS'}
SHELTER BAR
{'name': 'SHELTER BAR', 'category': 'BARS'}
SIDE BAR
{'name': 'SIDE BAR', 'category': 'BARS'}
MINIGOLF LISBON
{'name': 'MINIGOLF LISBON', 'category': 'BARS'}
XXL LISBON CLUB
{'name': 'XXL LISBON CLUB', 'category': 'BARS'}
DARK BY NUDE – ALBUFEIRA
{'name': 'DARK BY NUDE – ALBUFEIRA', 'category': 'BARS'}
THE FOREST – ALBUFEIRA
{'name': 'THE FORES

## Creating dataframe from the list

In [7]:
# cleaning the dataframe
categories_mapping = {'NIGHTCLUBS & PARTIES' : 'nightclub/bar', 'BARS': 'nightclub/bar', 'SAUNAS' : 'sauna', 'RESTAURANTS' : 'restaurant', 'LGBT COMMUNITY' : 'association', 'ACCOMMODATION' : 'accommodation', 'BEACHES' : 'beach'}

# standardizing the categories
df['category'] = df['category'].replace(categories_mapping);

# adding the city of Lisbon and country of Portugal for greater accuracy
df['partial_address'] = df['name'] + ', Lisboa - Portugal'

# website and ref
df['ref'] = 'https://www.lisbongaycircuit.com/#'
df['website'] = ''

# saving the dataframe
df.to_csv('lisbon_places.csv', index=False, encoding='utf-8-sig')
df

,name,category,partial_address,ref,website
0,TRUMPS,nightclub/bar,"TRUMPS, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,
1,FINALMENTE CLUB,nightclub/bar,"FINALMENTE CLUB, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,
2,CONSTRUCTION LISBON CLUB,nightclub/bar,"CONSTRUCTION LISBON CLUB, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,
3,PRESTIGE DANCECLUB – FARO,nightclub/bar,"PRESTIGE DANCECLUB – FARO, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,
4,106,nightclub/bar,"106, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,
5,BAR CRU,nightclub/bar,"BAR CRU, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,
6,CONSTRUCTION BAR – BAIRRO ALTO,nightclub/bar,"CONSTRUCTION BAR – BAIRRO ALTO, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,
7,ENTRETANTO ROOFTOP BAR,nightclub/bar,"ENTRETANTO ROOFTOP BAR, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,
8,TR3S,nightclub/bar,"TR3S, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,
9,SHELTER BAR,nightclub/bar,"SHELTER BAR, Lisboa - Portugal",https://www.lisbongaycircuit.com/#,


# Fim
Abaixo são só alguns testes

In [8]:
#lidando com redirecionamento para página
try:
    seletor_link_css = "a[href='https://www.timeout.pt/lisboa/pt/noite/os-melhores-bares-gay-de-lisboa']"

    link_redirecionamento = WebDriverWait(navegador, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, seletor_link_css))
    )
    link_redirecionamento.click()
except Exception as e:
    print("Problema no redirecionamento da página")

Problema no redirecionamento da página


In [9]:
#lidando com pop-up
try:
    id_fechar = (By.ID, "popup-newsletter-dismiss-button")
    
    # Espere até 5-10 segundos para o botão ser clicável
    botao_fechar = WebDriverWait(navegador, 10).until(
        EC.element_to_be_clickable(id_fechar)
    )
    
    botao_fechar.click()
    print("Pop-up newsletter fechado")
except Exception as e:
    # Se o botão não for encontrado, não faz nada e continua. O script não quebra.
    print("Nenhum pop-up encontrado ou não foi possível fechá-lo")

Nenhum pop-up encontrado ou não foi possível fechá-lo


In [10]:
# os nomes dos bares estão dentro de tag's h3 de classe específica
lista_div = navegador.find_elements(By.CSS_SELECTOR, 'div._title_osmln_9')
links = []
for d in lista_div:
    try:
        link = d.find_element(By.TAG_NAME, 'a').get_attribute('href')
        if link:
            links.append(link)
            #print(link)
    except Exception as e:
        print(f'Algum nome de bar está sem link. Erro:{e}')
#print(f"Todos os links: {links}")

NameError: name 'navegador' is not defined

In [ ]:
"""def collect_beaches_data(browser):
    url = "https://www.lisbongaycircuit.com/praias-lgbt-friendly-em-portugal/"
    browser.get(url)
    beaches = []
    try:
        div = browser.find_element(By.ID, 'single_count')
        paragraphs = div.find_elements(By.TAG_NAME, 'p')
        for p in paragraphs:
            print(p.text)
            if p.text:
                dictionary = {
                    'Name': p.text,
                    'Categorie': "Beaches"
                }
                beaches.append(dictionary)
                print(dictionary)
    except Exception as e:
        print(e)
    return beaches  
def collect_from_main_site(browser):
    data = []
    skip_categories =  ["Beaches", "Contacts & Agenda", "Shops & Services", "Tours & Travel"]
    try:
        selector_categorie = 'li.menu-item-has-children'   
        categories = WebDriverWait(browser, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, selector_categorie))
        )
        actions = ActionChains(browser)
        for i in categories:
            try:
                categorie_name = i.find_element(By.XPATH, './a').text
                if not categorie_name or categorie_name in skip_categories:
                    continue
            except Exception as e:
                print(f"AVISO: Uma categoria individual não pôde ser lida. Erro: {e}") 
                continue

            try:
                actions.move_to_element(i).perform()
                
                selector_item = ".//ul[contains(@class, 'sub-menu')]//li"
                WebDriverWait(i, 2).until(EC.visibility_of_element_located((By.XPATH, selector_item)))

            except Exception as e:
                print(e)
                continue
            
            selector_itens_submenu = ".//ul[contains(@class, 'sub-menu')]//a"
            itens_submenu = i.find_elements(By.XPATH, selector_itens_submenu)
            
            for j in itens_submenu:
                place_name = j.text
                print(place_name)
                if place_name:
                    dictionary = {
                        'Name': place_name,
                        'Categorie': categorie_name
                    }
                    data.append(dictionary)
                    print(dictionary)
    except Exception as e:
        print(f"Ocorreu um erro crítico durante a extração: {e}")
    return data"""